# Kizzasi Getting Started

**Kizzasi** (兆し, "sign/omen/premonition") is a Rust-native Autoregressive General-Purpose Signal Predictor (AGSP).

It uses State Space Models (Mamba, Mamba2, RWKV, S4D) with neuro-symbolic constraint enforcement to predict continuous signal streams: audio, sensors, video, control signals — all treated as equivalent signal streams.

This notebook walks through:
1. Installation and import
2. Basic signal generation (sine wave)
3. Configuring a predictor (Mamba2)
4. Single-step and multi-step autoregressive prediction
5. Plotting results

## Installation

```bash
pip install kizzasi
```

Kizzasi is a compiled Rust extension (via maturin/PyO3). Wheels are available for Linux, macOS, and Windows on PyPI.

## Imports

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt

# Try to import kizzasi; fall back to a pure-numpy stub for environments
# where the compiled wheel is not yet installed.
try:
    import kizzasi
    KIZZASI_AVAILABLE = True
    print(f'kizzasi version: {kizzasi.__version__}')
except ImportError:
    KIZZASI_AVAILABLE = False
    print('kizzasi not installed — running in numpy-stub mode')

print(f'numpy version: {np.__version__}')

## 1. Generate a Test Signal

We create a simple 440 Hz sine wave at 8000 Hz sample rate — a classic audio test signal.

In [ ]:
SAMPLE_RATE = 8000   # Hz
DURATION    = 0.5    # seconds
FREQ_HZ     = 440.0  # A4

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)
signal = np.sin(2 * np.pi * FREQ_HZ * t).astype(np.float32)

print(f'Signal shape: {signal.shape}  dtype: {signal.dtype}')
print(f'Value range : [{signal.min():.3f}, {signal.max():.3f}]')

## 2. Configure the Predictor

`kizzasi.Config` describes the SSM architecture. The `audio()` preset creates a single-channel predictor optimised for audio streams:

| Field           | Value  | Meaning                          |
|-----------------|--------|----------------------------------|
| `input_dim`     | 1      | single channel (mono audio)      |
| `output_dim`    | 1      | predict the next sample          |
| `hidden_dim`    | 256    | SSM hidden state width           |
| `num_layers`    | 4      | stacked SSM layers               |
| `state_dim`     | 16     | state-space latent dimension     |
| `context_window`| 8192   | maximum receptive field          |
| `model_type`    | mamba2 | architecture (Mamba2 / SSD)      |

In [ ]:
if KIZZASI_AVAILABLE:
    # High-level audio preset
    config = kizzasi.Config.audio(sample_rate=SAMPLE_RATE)
    print(config)

    # Or construct manually with full control:
    # config = kizzasi.Config(
    #     input_dim=1, output_dim=1,
    #     hidden_dim=128, num_layers=2,
    #     state_dim=8, context_window=4096,
    #     model_type='mamba2'
    # )
else:
    print('(stub) Config.audio(8000) -> input_dim=1, output_dim=1, model_type=mamba2')

## 3. Create a Predictor and Warm It Up

`kizzasi.Predictor` holds the SSM weights and hidden state. The state starts at zero and is updated on every `step()` call in O(1) time — no quadratic attention cost.

In [ ]:
WARMUP_STEPS = 200  # feed signal prefix before forecasting

if KIZZASI_AVAILABLE:
    predictor = kizzasi.Predictor(config)
    print(predictor)

    # Warm up: feed the first WARMUP_STEPS samples so the hidden state
    # captures the signal's periodicity before we start forecasting.
    for i in range(WARMUP_STEPS):
        x = np.array([signal[i]], dtype=np.float32)
        _ = predictor.step(x)

    print(f'Warm-up complete ({WARMUP_STEPS} steps)')
else:
    print('(stub) Predictor created and warmed up')

## 4. Autoregressive Forecasting

After warm-up we forecast `N_FORECAST` future samples autoregressively:
each prediction is fed back as the next input.

In [ ]:
N_FORECAST = 200

if KIZZASI_AVAILABLE:
    # predict_n() is a convenience wrapper that runs the autoregressive loop
    # entirely in Rust for maximum throughput.
    seed = np.array([signal[WARMUP_STEPS]], dtype=np.float32)
    forecast = predictor.predict_n(seed, n_steps=N_FORECAST)  # shape: (N_FORECAST, 1)
    forecast = forecast[:, 0]  # flatten to 1-D
else:
    # Numpy stub: continue the sine wave analytically
    t_fc = np.arange(N_FORECAST) / SAMPLE_RATE + WARMUP_STEPS / SAMPLE_RATE
    forecast = np.sin(2 * np.pi * FREQ_HZ * t_fc).astype(np.float32)

print(f'Forecast shape: {forecast.shape}')
print(f'Forecast range: [{forecast.min():.3f}, {forecast.max():.3f}]')

## 5. Plot: Ground Truth vs Prediction

In [ ]:
ground_truth = signal[WARMUP_STEPS : WARMUP_STEPS + N_FORECAST]
steps = np.arange(N_FORECAST)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Top: overlay
axes[0].plot(steps, ground_truth, label='Ground truth', color='steelblue', linewidth=1.5)
axes[0].plot(steps, forecast,     label='Kizzasi forecast', color='coral',
             linewidth=1.5, linestyle='--')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('440 Hz Sine Wave — Kizzasi Mamba2 Autoregressive Forecast')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: prediction error
error = ground_truth - forecast
axes[1].plot(steps, error, color='mediumseagreen', linewidth=1)
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle=':')
axes[1].set_ylabel('Error')
axes[1].set_xlabel('Step')
axes[1].set_title(f'Prediction Error  (MAE={np.abs(error).mean():.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kizzasi_getting_started.png', dpi=120)
plt.show()
print('Plot saved to /tmp/kizzasi_getting_started.png')

## 6. Other Configuration Presets

Kizzasi ships with presets for common domains.

In [ ]:
if KIZZASI_AVAILABLE:
    # 6-DOF robot arm: 6 joint angles in, 6 actions out
    robotics_cfg = kizzasi.Config.robotics(state_dim=6, action_dim=6)
    print('Robotics:', robotics_cfg)

    # 9-axis IMU sensor fusion
    sensor_cfg = kizzasi.Config.sensor(num_sensors=9)
    print('Sensor  :', sensor_cfg)

    # Tiny edge model (32 hidden, 1 layer)
    edge_cfg = kizzasi.Config.lightweight(input_dim=2, output_dim=2)
    print('Edge    :', edge_cfg)
else:
    print('(stub) Config presets: robotics, sensor, lightweight')

## 7. Resetting State

Call `predictor.reset()` to zero out the hidden state — useful when switching to a new signal segment.

In [ ]:
if KIZZASI_AVAILABLE:
    predictor.reset()
    print('Hidden state zeroed.')

    # step_list() accepts plain Python lists if you prefer to avoid numpy
    out = predictor.step_list([0.0])   # input_dim=1
    print(f'step_list([0.0]) -> {out}')
else:
    print('(stub) predictor.reset() and step_list() demonstrated')

## Summary

| API | Purpose |
|-----|---------|
| `kizzasi.Config(...)` / presets | Configure architecture |
| `kizzasi.Predictor(config)` | Instantiate model |
| `predictor.step(np_array)` | Single O(1) step, returns numpy |
| `predictor.step_list(list)` | Single step, pure Python lists |
| `predictor.predict_n(seed, n)` | Autoregressive N-step forecast |
| `predictor.reset()` | Zero hidden state |
| `predictor.set_guardrails([...])` | Attach constraint guardrails |

Next: see `audio_processing.ipynb` for MFCC features and streaming tokenization, or `anomaly_detection.ipynb` for constraint-guided anomaly detection.